In [1]:
import pandas as pd

/Users/oceane/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/oceane/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Data analysis

In [2]:
reviews = pd.read_csv("../Data/sitdown_reviews_25mb.csv")

reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,name,city,state,restaurant_stars
0,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,Kettle Restaurant,Tucson,AZ,3.5
1,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,Zaika,Philadelphia,PA,4.0
2,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1.0,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,Dmitri's,Philadelphia,PA,4.0
3,pUycOfUwM8vqX7KjRRhUEA,59MxRhNVhU9MYndMkz0wtw,gebiRewfieSdtt17PTW6Zg,3.0,0,0,0,Had a party of 6 here for hibachi. Our waitres...,2016-07-25 07:31:06,Hibachi Steak House & Sushi Bar,Santa Barbara,CA,3.5
4,8JFGBuHMoiNDyfcxuWNtrA,smOvOajNG0lS4Pq7d8g4JQ,RZtGWDLCAtuipwaZ-UfjmQ,4.0,0,0,0,Good food--loved the gnocchi with marinara\nth...,2009-10-14 19:57:14,LaScala's,Philadelphia,PA,3.5


In [3]:
print("Shape:", reviews.shape)
print("Number of reviews:", len(reviews))
print("Number of unique restaurants:", reviews["business_id"].nunique())

Shape: (32000, 13)
Number of reviews: 32000
Number of unique restaurants: 2922


Cutting into sentences

In [4]:
import nltk
from nltk.tokenize import sent_tokenize

In [5]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /Users/oceane/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/oceane/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
test_review = reviews.loc[0, "text"]

print("Original review:")
print(test_review)

print("\nSentences:")
sent_tokenize(test_review)

Original review:
Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with no expectations. Next to the Clarion Hotel.

Sentences:


['Family diner.',
 'Had the buffet.',
 'Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon.',
 'All good.',
 'Lots of Mexican choices there.',
 'Also has a menu with breakfast served all day long.',
 'Friendly, attentive staff.',
 'Good place for a casual relaxed meal with no expectations.',
 'Next to the Clarion Hotel.']

In [7]:
reviews["sentences"] = reviews["text"].apply(sent_tokenize)

reviews[["review_id", "text", "sentences"]].head()

,review_id,text,sentences
0,saUsX_uimxRlCVr67Z4Jig,Family diner. Had the buffet. Eclectic assortm...,"[Family diner., Had the buffet., Eclectic asso..."
1,AqPFMleE6RsU23_auESxiA,"Wow! Yummy, different, delicious. Our favo...","[Wow!, Yummy, different, delicious., Our favo..."
2,JrIxlS1TzJ-iCu79ul40cQ,I am a long term frequent customer of this est...,[I am a long term frequent customer of this es...
3,pUycOfUwM8vqX7KjRRhUEA,Had a party of 6 here for hibachi. Our waitres...,"[Had a party of 6 here for hibachi., Our waitr..."
4,8JFGBuHMoiNDyfcxuWNtrA,Good food--loved the gnocchi with marinara\nth...,[Good food--loved the gnocchi with marinara\nt...


Aspect dictionary

In [8]:
aspect_keywords = {
    "quality": [
        "food", "dish", "dishes",
        "taste", "flavor", "flavors",
        "portion", "portions",
        "ingredient", "ingredients",
        "entree", "entrees",
        "appetizer", "appetizers",
        "dessert", "desserts"
    ],

    "price": [
        "price", "prices", "pricing",
        "cost", "costs",
        "value", "expensive", "cheap",
        "overpriced", "affordable"
    ],

    "convenience": [
        "parking", "location",
        "wait", "waiting",
        "reservation", "reservations",
        "accessible", "access",
        "delivery", "takeout", "pickup"
    ]
}

In [9]:
import re

def identify_aspects(sentence):
    sentence_lower = sentence.lower()
    matched_aspects = []

    for aspect, keywords in aspect_keywords.items():
        for keyword in keywords:
            pattern = r"\b" + re.escape(keyword) + r"\b"
            if re.search(pattern, sentence_lower):
                matched_aspects.append(aspect)
                break

    return matched_aspects

In [10]:
sentences_df = reviews[
    ["review_id", "business_id", "restaurant_stars", "sentences"]
].explode("sentences").rename(columns={"sentences": "sentence"})

sentences_df.head()

,review_id,business_id,restaurant_stars,sentence
0,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3.5,Family diner.
0,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3.5,Had the buffet.
0,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3.5,"Eclectic assortment: a large chicken leg, frie..."
0,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3.5,All good.
0,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3.5,Lots of Mexican choices there.


In [11]:
sentences_df["aspects"] = sentences_df["sentence"].apply(identify_aspects)

pd.set_option("display.max_colwidth", None)

sentences_df[sentences_df["aspects"].apply(len) > 0][
    ["sentence", "aspects"]
].sample(20, random_state=42)

,sentence,aspects
8446,"It's not that Samurai is the absolute worst sushi in Nashville (you can go to Ken's for that), it's just that the food as a whole is really really mehhh.",[quality]
16929,What I can't get over about this place is the price of the food.,"[quality, price]"
5083,A great atmosphere and location in Germantown.,[convenience]
25764,"As long as you like heat, you'll love this dish.",[quality]
2286,The portion size vs the price was reasonable.,"[quality, price]"
22181,Food is good.,[quality]
27757,"In addition, one of our party ordered the duck, which wasn't ready when the other dishes were.",[quality]
2566,"I have been on the Tony Roma's birthday club for the last few years so this year, I decided to try this location.",[convenience]
20196,At the very least all the food is very flavorful.,[quality]
7788,"We got our food very quickly, and he checked in on us quite often.",[quality]


VADER

In [12]:
nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/oceane/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [13]:
price_tests = [
    "The food was expensive.",
    "The food was cheap.",
    "The price was affordable.",
    "The price was overpriced.",
    "The price was reasonable.",
    "The food was worth the price.",
    "Great value for the money.",
    "Not worth the money."
]

for sentence in price_tests:
    print(sentence, "->", sia.polarity_scores(sentence)["compound"])

The food was expensive. -> 0.0
The food was cheap. -> 0.0
The price was affordable. -> 0.0
The price was overpriced. -> 0.0
The price was reasonable. -> 0.0
The food was worth the price. -> 0.2263
Great value for the money. -> 0.7579
Not worth the money. -> -0.1695


In [14]:
# update
price_lexicon = {
    "expensive": -2.0,
    "overpriced": -2.5,
    "affordable": 2.0,
    "reasonable": 1.5
}

sia.lexicon.update(price_lexicon)

In [15]:
aspect_sentences = sentences_df[
    sentences_df["aspects"].apply(len) > 0
].copy()

print("Number of sentences left:", len(aspect_sentences))

Number of sentences left: 50588


In [16]:
aspect_sentences["sentiment"] = aspect_sentences["sentence"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

aspect_sentences[
    ["business_id", "sentence", "aspects", "sentiment"]
].head(20)

,business_id,sentence,aspects,sentiment
3,gebiRewfieSdtt17PTW6Zg,"Service was fishy, food was pretty good, and im hoping it was just an off night here.",[quality],0.8360
4,RZtGWDLCAtuipwaZ-UfjmQ,"Good food--loved the gnocchi with marinara\nthe baked eggplant appetizer was very good too\n\nThe service was very slow, but despite this, I'd go back, the food is just that good",[quality],0.8093
5,otQS34_MymijPTdNBoBdCw,Super cheap and you can drive through.,[price],0.5994
9,cPepkJeRMtHapc_b2Oe_dw,The choices particularly of the vegetables seemed a little too limiting and I felt I had more rice than other food items.,[quality],0.0000
9,cPepkJeRMtHapc_b2Oe_dw,My husband ordered the Maui roll which is my favorite roll at the 96th street location and this just was not up that standard.,[convenience],0.4588
9,cPepkJeRMtHapc_b2Oe_dw,"As a healthy alternative to fast food in the area, it is worth the wait.","[quality, convenience]",0.5574
10,x4XdNhp0Xn8lOivzc77J-g,Best thai food in the area.,[quality],0.6369
11,S2Ho8yLxhKAa26pBAm6rxA,"Service was crappy, and food was mediocre.",[quality],-0.5574
12,MWmXGQ98KbRo3vsS5nZhMA,My wife and I were astounded by how quickly our food came out!,[quality],0.4753
12,MWmXGQ98KbRo3vsS5nZhMA,Can't wait to see what the next chef specials will be!,[convenience],0.0000


In [17]:
# sanity check
aspect_sentences[
    ["sentence", "aspects", "sentiment"]
].sample(20, random_state=42)

,sentence,aspects,sentiment
8446,"It's not that Samurai is the absolute worst sushi in Nashville (you can go to Ken's for that), it's just that the food as a whole is really really mehhh.",[quality],-0.6249
16929,What I can't get over about this place is the price of the food.,"[quality, price]",0.0000
5083,A great atmosphere and location in Germantown.,[convenience],0.6249
25764,"As long as you like heat, you'll love this dish.",[quality],0.7717
2286,The portion size vs the price was reasonable.,"[quality, price]",0.3612
22181,Food is good.,[quality],0.4404
27757,"In addition, one of our party ordered the duck, which wasn't ready when the other dishes were.",[quality],0.1506
2566,"I have been on the Tony Roma's birthday club for the last few years so this year, I decided to try this location.",[convenience],0.0000
20196,At the very least all the food is very flavorful.,[quality],0.0000
7788,"We got our food very quickly, and he checked in on us quite often.",[quality],0.0000


In [18]:
aspect_expanded = aspect_sentences.explode("aspects").copy()

aspect_expanded[
    ["business_id", "sentence", "aspects", "sentiment"]
].head(20)

,business_id,sentence,aspects,sentiment
3,gebiRewfieSdtt17PTW6Zg,"Service was fishy, food was pretty good, and im hoping it was just an off night here.",quality,0.8360
4,RZtGWDLCAtuipwaZ-UfjmQ,"Good food--loved the gnocchi with marinara\nthe baked eggplant appetizer was very good too\n\nThe service was very slow, but despite this, I'd go back, the food is just that good",quality,0.8093
5,otQS34_MymijPTdNBoBdCw,Super cheap and you can drive through.,price,0.5994
9,cPepkJeRMtHapc_b2Oe_dw,The choices particularly of the vegetables seemed a little too limiting and I felt I had more rice than other food items.,quality,0.0000
9,cPepkJeRMtHapc_b2Oe_dw,My husband ordered the Maui roll which is my favorite roll at the 96th street location and this just was not up that standard.,convenience,0.4588
9,cPepkJeRMtHapc_b2Oe_dw,"As a healthy alternative to fast food in the area, it is worth the wait.",quality,0.5574
9,cPepkJeRMtHapc_b2Oe_dw,"As a healthy alternative to fast food in the area, it is worth the wait.",convenience,0.5574
10,x4XdNhp0Xn8lOivzc77J-g,Best thai food in the area.,quality,0.6369
11,S2Ho8yLxhKAa26pBAm6rxA,"Service was crappy, and food was mediocre.",quality,-0.5574
12,MWmXGQ98KbRo3vsS5nZhMA,My wife and I were astounded by how quickly our food came out!,quality,0.4753


In [19]:
aspect_summary = (
    aspect_expanded
    .groupby(["business_id", "aspects"])
    .agg(
        sentence_count=("sentence", "count"),
        mean_sentiment=("sentiment", "mean")
    )
    .reset_index()
)

aspect_summary.head(20)

,business_id,aspects,sentence_count,mean_sentiment
0,-0Ym1Wg3bXd_TDz8JtvOQg,convenience,2,0.492300
1,-0Ym1Wg3bXd_TDz8JtvOQg,price,1,0.425300
2,-0Ym1Wg3bXd_TDz8JtvOQg,quality,7,0.221771
3,-0fvhILrC9UsQ6gLNpZlTQ,convenience,1,0.177900
4,-0fvhILrC9UsQ6gLNpZlTQ,quality,1,0.831600
5,-2Axhv9AZ_n7qjQefECpVw,price,1,0.361200
6,-2Axhv9AZ_n7qjQefECpVw,quality,4,0.073175
7,-2bLM6nIETD4Mk-rhjIYZQ,quality,1,0.750600
8,-49YlJ2GGzgmQ71FX9gGyg,convenience,3,0.392967
9,-49YlJ2GGzgmQ71FX9gGyg,quality,2,0.000000


In [20]:
# fliter out the restaurant-aspect combinations that has less than 3 sentences
aspect_summary_filtered = aspect_summary[
    aspect_summary["sentence_count"] >= 3
].copy()

print("Restaurant-aspect combinations before filtering:",
      len(aspect_summary))

print("Restaurant-aspect combinations after filtering:",
      len(aspect_summary_filtered))

Restaurant-aspect combinations before filtering: 6101
Restaurant-aspect combinations after filtering: 3582


In [21]:
restaurant_features = (
    aspect_summary_filtered
    .pivot(
        index="business_id",
        columns="aspects",
        values="mean_sentiment"
    )
    .reset_index()
)

restaurant_features.head(20)

aspects,business_id,convenience,price,quality
0,-0Ym1Wg3bXd_TDz8JtvOQg,NaN,NaN,0.221771
1,-2Axhv9AZ_n7qjQefECpVw,NaN,NaN,0.073175
2,-49YlJ2GGzgmQ71FX9gGyg,0.392967,NaN,NaN
3,-4dYswJy7SPcbcERvitmIg,NaN,-0.208600,NaN
4,-6OjnX3ZdDOhHxWR60wysg,NaN,0.683633,0.290014
5,-JStjL-8mRZq4ov4uI1FaQ,NaN,0.410840,0.364648
6,00Vn4VUTWl71_cFPziRQNQ,NaN,NaN,0.640125
7,04UD14gamNjLY0IDYVhHJg,0.207976,0.302920,0.278390
8,07i5EdI8v2FBhVV-rya8Wg,0.228367,0.315033,0.483770
9,09za1h1fdeF9rTSviTV3zg,0.251477,NaN,0.445396


In [22]:
restaurant_features_complete = restaurant_features.dropna(
    subset=["quality", "price", "convenience"]
).copy()

print("Restaurants before complete-case filtering:",
      len(restaurant_features))

print("Restaurants after complete-case filtering:",
      len(restaurant_features_complete))

restaurant_features_complete.head(20)

Restaurants before complete-case filtering: 1924
Restaurants after complete-case filtering: 539


aspects,business_id,convenience,price,quality
7,04UD14gamNjLY0IDYVhHJg,0.207976,0.302920,0.278390
8,07i5EdI8v2FBhVV-rya8Wg,0.228367,0.315033,0.483770
10,0AQJ-QTu7GtxaP4SoD07yA,-0.137900,0.026033,0.455853
15,0JPi9cyV9i3_kDPj049_qQ,0.131367,0.201660,0.462536
16,0JoB3ThhI-DS78-1Ks1EUQ,0.323414,0.382008,0.180057
17,0Kn5W22UmxOqPj2cjouFNA,0.047500,0.154800,0.120733
25,0hIXH9jMdHov1VrLC8ujUg,0.084567,0.406300,0.466684
29,0sr1EyOc6Td1C-962QW88w,0.294183,0.416825,0.174880
33,0wQCEcpZ57TmTm6EmEDsIw,0.158953,0.066900,0.208358
36,0z-pNv_92L_-LiYV_XlcSw,0.054400,0.523400,0.251660


Train

In [23]:
restaurant_ratings = (
    reviews[["business_id", "restaurant_stars"]]
    .drop_duplicates(subset="business_id")
)

model_data = restaurant_features_complete.merge(
    restaurant_ratings,
    on="business_id",
    how="left"
)

print("Shape of modeling dataset:", model_data.shape)

print("\nMissing values:")
print(model_data.isna().sum())

model_data.head(20)

Shape of modeling dataset: (539, 5)

Missing values:
business_id         0
convenience         0
price               0
quality             0
restaurant_stars    0
dtype: int64


,business_id,convenience,price,quality,restaurant_stars
0,04UD14gamNjLY0IDYVhHJg,0.207976,0.302920,0.278390,4.0
1,07i5EdI8v2FBhVV-rya8Wg,0.228367,0.315033,0.483770,4.5
2,0AQJ-QTu7GtxaP4SoD07yA,-0.137900,0.026033,0.455853,4.5
3,0JPi9cyV9i3_kDPj049_qQ,0.131367,0.201660,0.462536,4.0
4,0JoB3ThhI-DS78-1Ks1EUQ,0.323414,0.382008,0.180057,3.5
5,0Kn5W22UmxOqPj2cjouFNA,0.047500,0.154800,0.120733,3.0
6,0hIXH9jMdHov1VrLC8ujUg,0.084567,0.406300,0.466684,4.5
7,0sr1EyOc6Td1C-962QW88w,0.294183,0.416825,0.174880,4.0
8,0wQCEcpZ57TmTm6EmEDsIw,0.158953,0.066900,0.208358,3.5
9,0z-pNv_92L_-LiYV_XlcSw,0.054400,0.523400,0.251660,3.5


In [24]:
from sklearn.model_selection import train_test_split

X = model_data[["quality", "price", "convenience"]]
y = model_data["restaurant_stars"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Total restaurants:", len(model_data))
print("Training restaurants:", len(X_train))
print("Testing restaurants:", len(X_test))

Total restaurants: 539
Training restaurants: 431
Testing restaurants: 108


In [25]:
# fit multiple linear regression model
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Intercept:", model.intercept_)

for feature, coefficient in zip(X.columns, model.coef_):
    print(feature, "coefficient:", coefficient)

Intercept: 3.100256300044712
quality coefficient: 1.595114628232512
price coefficient: 0.2341771448386729
convenience coefficient: 0.32051561259964007


Test

In [26]:
y_pred = model.predict(X_test)

predictions = pd.DataFrame({
    "actual_stars": y_test,
    "predicted_stars": y_pred
})

predictions.head(20)

,actual_stars,predicted_stars
486,3.5,3.696025
73,4.0,3.982753
349,4.0,3.469293
86,3.5,3.634357
457,2.5,3.546051
77,3.0,3.423572
512,4.5,3.716927
374,4.5,4.153626
89,4.0,3.930097
429,3.5,3.799441


Evaluation

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.4119259736225736
RMSE: 0.5071244877232091
R²: 0.2640598454587776
